# Mạng nơ-ron: forward, backward và cập nhật trong cùng một trạng thái

Mở ngày 19/20/21 trên web hoặc tab **Mạng nơ-ron** trong dự án giá nhà. Chọn một mẫu và một cạnh. Viết dự đoán trước khi tiến bước.

Mạng có 2 đầu vào, một lớp ẩn tanh (hoặc tuyến tính để so sánh), đầu ra sigmoid và loss BCE trung bình. Dữ liệu XOR/hai vành trăng là tổng hợp; mọi metric dưới đây đo trên tập minh họa huấn luyện, chưa phải validation/test.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def init_network(hidden=4,seed=42):
    state=seed%(2**32)
    def random():
        nonlocal state
        state=(1664525*state+1013904223)%(2**32)
        return state/(2**32)
    return {'W':np.array([[(2*random()-1)*.9 for _ in range(hidden)] for _ in range(2)]),'b':np.array([(2*random()-1)*.15 for _ in range(hidden)]),'v':np.array([(2*random()-1)/np.sqrt(hidden) for _ in range(hidden)]),'c':0.}
def copy_parameters(p):return {k:np.array(v,copy=True) if k!='c' else float(v) for k,v in p.items()}
def forward(p,X,y,activation='tanh'):
    z=X@p['W']+p['b'];a=np.tanh(z) if activation=='tanh' else z
    s=a@p['v']+p['c'];prob=np.exp(-np.logaddexp(0,-s));loss=np.logaddexp(0,s)-y*s
    return {'z':z,'a':a,'s':s,'p':prob,'row_loss':loss,'loss':float(loss.mean())}
def backward(p,X,y,activation='tanh'):
    f=forward(p,X,y,activation);dout=(f['p']-y)/len(y)
    dh=dout[:,None]*p['v']*(1-f['a']**2 if activation=='tanh' else 1)
    g={'W':X.T@dh,'b':dh.sum(axis=0),'v':f['a'].T@dout,'c':float(dout.sum())}
    return g,dout,dh
def updated(p,g,rate):return {k:p[k]-rate*g[k] for k in p}
X=np.array([[-1.,-1],[-1,1],[1,-1],[1,1]]);y=np.array([0,1,1,0.]);parameters=init_network()
print('Mạng có',len(parameters['b']),'nút ẩn. BCE trung bình dùng N=',len(y))

## 1. Forward của một mẫu

Các trọng số chưa đổi. Thử đổi `sample` hoặc chạy ô này sau khi tự sửa một trọng số. Logit s khác xác suất p; activation lớp ẩn khác activation đầu ra.

In [ ]:
sample=0
activation='tanh'
f=forward(parameters,X,y,activation)
display(pd.DataFrame({'x1*W1':X[sample,0]*parameters['W'][0],'x2*W2':X[sample,1]*parameters['W'][1],'b':parameters['b'],'z':f['z'][sample],'a':f['a'][sample]}))
print('logit =',f['s'][sample],'; p =',f['p'][sample],'; y =',y[sample])
print('BCE của mẫu =',f['row_loss'][sample],'; BCE trung bình =',f['loss'])
fig,axs=plt.subplots(1,3,figsize=(10,3))
for ax,values,title in zip(axs,[parameters['W'],parameters['v'][None,:],f['a'][sample][None,:]],['W input to hidden','v hidden to output','Activation of selected sample']):
    im=ax.imshow(values,cmap='RdBu_r',aspect='auto');ax.set_title(title);fig.colorbar(im,ax=ax)
plt.tight_layout();plt.show()

## 2. Backward, chia N và kiểm tra gradient số

Với BCE và sigmoid, δđầu-ra=(p−y)/N. Mỗi đóng góp đã chia N; cộng từ các mẫu để có gradient trung bình, không chia thêm lần nữa. Backward phải dùng v cũ, trước khi cập nhật bất kỳ lớp nào.

In [ ]:
g,dout,dh=backward(parameters,X,y,activation)
display(pd.DataFrame({'p-y':f['p']-y,'delta_output_mean':dout}))
print('Gradient W:',g['W']);print('Gradient b:',g['b']);print('Gradient v:',g['v']);print('Gradient c:',g['c'])
checks=[]
for key,value in parameters.items():
    shape=np.shape(value)
    for index in (list(np.ndindex(shape)) if shape else [()]):
        plus=copy_parameters(parameters);minus=copy_parameters(parameters);eps=1e-5
        if shape:plus[key][index]+=eps;minus[key][index]-=eps;analytic=g[key][index]
        else:plus[key]+=eps;minus[key]-=eps;analytic=g[key]
        numeric=(forward(plus,X,y,activation)['loss']-forward(minus,X,y,activation)['loss'])/(2*eps)
        checks.append([key,str(index),analytic,numeric,abs(analytic-numeric)])
display(pd.DataFrame(checks,columns=['Parameter','Index','Analytic','Numeric','Absolute error']))
assert max(row[-1] for row in checks)<1e-7
print('Tất cả gradient khớp sai phân số.')

## 3. Cập nhật đồng thời và so tanh với lớp ẩn tuyến tính

Giữ dữ liệu, seed, learning rate và số nút; chỉ đổi activation. Không kết luận nhiều nút luôn tốt hơn. Tập XOR nhỏ giúp thấy giới hạn của ranh giới tuyến tính, chưa đánh giá khả năng tổng quát hóa.

In [ ]:
rate=.3;steps=150;histories={};final={}
for act in ['tanh','linear']:
    p=init_network();history=[]
    for step in range(steps+1):
        stats=forward(p,X,y,act);history.append(stats['loss'])
        if step<steps:
            gradient,_,_=backward(p,X,y,act)
            p=updated(p,gradient,rate)
    histories[act]=history;final[act]=p
fig,axs=plt.subplots(1,2,figsize=(10,4))
for act,history in histories.items():axs[0].plot(history,label=act)
axs[0].set(xlabel='True parameter updates',ylabel='Mean BCE');axs[0].legend()
a,b=np.meshgrid(np.linspace(-1.4,1.4,60),np.linspace(-1.4,1.4,60));grid=np.column_stack([a.ravel(),b.ravel()]);p=forward(final['tanh'],grid,np.zeros(len(grid)),'tanh')['p'].reshape(a.shape)
axs[1].contourf(a,b,p,levels=np.linspace(0,1,11),cmap='RdBu_r',alpha=.5);axs[1].contour(a,b,p,levels=[.5],colors='black');axs[1].scatter(X[:,0],X[:,1],c=y,cmap='RdBu_r',edgecolors='black');axs[1].set_title('Trained tanh network on XOR');plt.tight_layout();plt.show()
for act,p in final.items():
    stats=forward(p,X,y,act);print(act,'BCE:',stats['loss'],'training accuracy:',np.mean((stats['p']>=.5)==y))
assert histories['tanh'][-1]<histories['tanh'][0]
assert histories['linear'][-1]>.69

## 4. Ngưỡng chỉ thay nhãn dự đoán

Thử ngưỡng khác. BCE và xác suất của mô hình đã học không thay đổi vì chưa có bước cập nhật trọng số nào.

In [ ]:
stats=forward(final['tanh'],X,y,'tanh')
thresholds=[.2,.5,.8]
table=pd.DataFrame({'x1':X[:,0],'x2':X[:,1],'truth':y,'probability':stats['p']})
for threshold in thresholds:table['label@'+str(threshold)]=(stats['p']>=threshold).astype(int)
display(table)
print('Cùng một BCE:',stats['loss'])

## 5. Đối chiếu bản ghi từ web

Bấm **Tải bản ghi mạng đã quan sát**, đưa JSON vào thư mục notebook và sửa TRACE_PATH. Bản ghi có thể dừng ở giữa một lượt. Kiểm tra bên dưới tính lại forward, gradient mỗi mẫu và cập nhật bằng NumPy cho mọi trạng thái đã xuất.

In [ ]:
TRACE_PATH=''  # ví dụ 'trace-neural-network.json'
def audit_network(trace):
    assert trace['algorithm']=='neural-network'
    rows=trace['samples'];X=np.array([r['x'] for r in rows]);y=np.array([r['y'] for r in rows]);act=trace['activation'];rate=trace['rate']
    expected_init=init_network(trace['hidden'],trace['seed'])
    for key in expected_init:assert np.allclose(expected_init[key],trace['frames'][0]['before'][key])
    for frame in trace['frames']:
        before={k:np.asarray(v,float) if k!='c' else float(v) for k,v in frame['before'].items()}
        stats=forward(before,X,y,act);gradient,dout,dh=backward(before,X,y,act)
        for key in gradient:assert np.allclose(gradient[key],frame['gradient']['total'][key],atol=1e-9)
        for i,row in enumerate(frame['gradient']['rows']):
            assert np.isclose(row['deltaOutput'],dout[i]) and np.allclose(row['deltaHidden'],dh[i])
            assert np.allclose(row['W'],X[i,:,None]*dh[i]) and np.allclose(row['v'],stats['a'][i]*dout[i])
        shown=updated(before,gradient,rate) if frame['phase']=='update' and not frame['stopped'] else before
        for key in shown:assert np.allclose(shown[key],frame['parameters'][key],atol=1e-9)
        actual=forward(shown,X,y,act)
        assert np.isclose(actual['loss'],frame['stats']['loss'])
        for i,row in enumerate(frame['stats']['rows']):
            assert np.allclose(actual['z'][i],row['z']) and np.allclose(actual['a'][i],row['a'])
            assert np.isclose(actual['s'][i],row['logit']) and np.isclose(actual['p'][i],row['p'])
    print('Khớp NumPy:',len(trace['frames']),'trạng thái;',trace['dataset'],act,'hidden=',trace['hidden'])
if TRACE_PATH:audit_network(json.loads(Path(TRACE_PATH).read_text()))
else:print('Chưa chọn trace; các phần trên vẫn chạy độc lập.')

## Thực hành A/B/C

**A — tính tay:** Chọn một mẫu và một cạnh x₁→nút ẩn. Tính z, a, δđầu-ra, δẩn và đóng góp của mẫu vào gradient cạnh đó. Ghi rõ chỗ chia N.

**B — tự kiểm chứng:** Chọn một cạnh trên web, chạy tới backward, so gradient giải tích với sai phân số. Xuất JSON và chạy audit_network trong Python.

**C — mở rộng:** So tanh/tuyến tính, 2/4/6 nút, learning rate hoặc seed; chỉ đổi một yếu tố mỗi lượt. Dùng một tập validation độc lập trước khi chọn cấu hình cho dữ liệu thật.

Nhật ký: Em dự đoán dấu gradient thế nào? Cạnh nào đã đổi sau update? Loss/nhãn đổi vì trọng số hay vì ngưỡng? Mô phỏng này chưa chứng minh điều gì về dữ liệu chưa thấy?